In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim

from models import SourceEncoder, TargetEncoder

In [7]:
# Load the tensors
x_cat_tensor = torch.load('../dataset/processed/x_cat_tensor.pt')
x_ratio_tensor = torch.load('../dataset/processed/x_ratio_tensor.pt')
categorical_cardinalities = torch.load('../dataset/processed/cardinalities.pt').tolist()
labels_tensor = torch.load('../dataset/processed/labels_tensor.pt')

print("데이터 로드 완료:", x_cat_tensor.shape, ",", x_ratio_tensor.shape, ",", len(categorical_cardinalities))

데이터 로드 완료: torch.Size([14676, 23]) , torch.Size([14676, 4]) , 23


In [8]:
# Define PyTorch DataLoader
class UserDataset(Dataset):
    def __init__(self, x_cat_tensor, x_ratio_tensor, labels_tensor):
        self.x_cat = x_cat_tensor
        self.x_ratio = x_ratio_tensor
        self.labels = labels_tensor
        
    def __len__(self):
        return len(self.x_cat)
        
    def __getitem__(self, idx):
        # 비금융 데이터, 금융 데이터, 5단계 라벨을 함께 반환
        return self.x_cat[idx], self.x_ratio[idx], self.labels[idx]

# Create Dataset and DataLoader with the preprocessed tensors
dataset = UserDataset(x_cat_tensor, x_ratio_tensor, labels_tensor)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True) # Batch size is adjustable

In [9]:
def supcon_loss(z_NF, z_F, labels, temperature=0.1):
    """
    Supervised Contrastive Learning Loss
    - z_NF: Source domain embeddings (Batch, Dim)
    - z_F: Target domain embeddings (Batch, Dim)
    - labels: 5단계 위험 성향 라벨 (Batch,)
    """
    device = z_NF.device
    
    # 1. 코사인 유사도 계산을 위한 L2 정규화
    z_NF = F.normalize(z_NF, dim=1)
    z_F = F.normalize(z_F, dim=1)
    
    # 2. 유사도 행렬 계산 (Batch x Batch)
    similarity_matrix = torch.matmul(z_NF, z_F.T) / temperature
    
    # 3. 정답 마스크(Mask) 생성: 라벨이 같으면 1, 다르면 0
    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)
    
    # 4. Log Softmax로 확률값 계산 (수치적 안정성을 위해 log_softmax 사용)
    log_prob = F.log_softmax(similarity_matrix, dim=1)
    
    # 5. 동일 라벨을 가진 Positive 쌍들의 Log 확률 평균 계산
    # 각 Anchor당 존재하는 Positive 샘플의 개수로 나누어줌
    mask_sum = mask.sum(dim=1)
    # 0으로 나누는 것을 방지 (최소 1은 보장됨. 자기 자신 또는 동일 쌍이 있으므로)
    mask_sum = torch.max(mask_sum, torch.ones_like(mask_sum))
    
    # Positive 쌍에 해당하는 log_prob만 더한 뒤 평균 (Loss는 작아져야 하므로 음수 취함)
    mean_log_prob_pos = (mask * log_prob).sum(dim=1) / mask_sum
    loss = -mean_log_prob_pos.mean()
    
    return loss

In [10]:
# Set Training Loop

# Initialize models (using classes defined in encoding.ipynb)
source_encoder = SourceEncoder(categorical_cardinalities, embed_dim=16, output_dim=128)
target_encoder = TargetEncoder(input_dim=4, output_dim=128)

# Set up optimizer (training parameters for both encoders simultaneously)
optimizer = optim.Adam(list(source_encoder.parameters()) + list(target_encoder.parameters()), lr=0.0001)

num_epochs = 300 # number of training epochs

for epoch in range(num_epochs):
    source_encoder.train()
    target_encoder.train()
    
    total_loss = 0.0
    
    # dataloader에서 라벨(batch_labels)도 함께 받아옴
    for batch_x_cat, batch_x_ratio, batch_labels in dataloader:
        optimizer.zero_grad() # initialize gradients
        
        # pass through models
        z_NF = source_encoder(batch_x_cat)
        z_F = target_encoder(batch_x_ratio)
        
        # Calculate Loss
        loss = supcon_loss(z_NF, z_F, batch_labels, temperature=0.05)
        
        # Backward pass and weight update
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("Contrastive Learning Completed!")

# Save trained model weights (for later use in Stage 3)
torch.save(source_encoder.state_dict(), '../checkpoints/source_encoder.pth')
torch.save(target_encoder.state_dict(), '../checkpoints/target_encoder.pth')

Epoch [1/300], Loss: 6.5006
Epoch [2/300], Loss: 6.3281
Epoch [3/300], Loss: 6.2862
Epoch [4/300], Loss: 6.2635
Epoch [5/300], Loss: 6.2497
Epoch [6/300], Loss: 6.2416
Epoch [7/300], Loss: 6.2357
Epoch [8/300], Loss: 6.2308
Epoch [9/300], Loss: 6.2278
Epoch [10/300], Loss: 6.2255
Epoch [11/300], Loss: 6.2231
Epoch [12/300], Loss: 6.2220
Epoch [13/300], Loss: 6.2209
Epoch [14/300], Loss: 6.2194
Epoch [15/300], Loss: 6.2182
Epoch [16/300], Loss: 6.2174
Epoch [17/300], Loss: 6.2164
Epoch [18/300], Loss: 6.2170
Epoch [19/300], Loss: 6.2170
Epoch [20/300], Loss: 6.2157
Epoch [21/300], Loss: 6.2159
Epoch [22/300], Loss: 6.2149
Epoch [23/300], Loss: 6.2151
Epoch [24/300], Loss: 6.2144
Epoch [25/300], Loss: 6.2136
Epoch [26/300], Loss: 6.2134
Epoch [27/300], Loss: 6.2137
Epoch [28/300], Loss: 6.2127
Epoch [29/300], Loss: 6.2131
Epoch [30/300], Loss: 6.2124
Epoch [31/300], Loss: 6.2124
Epoch [32/300], Loss: 6.2118
Epoch [33/300], Loss: 6.2116
Epoch [34/300], Loss: 6.2123
Epoch [35/300], Loss: 6